# Data Panel Creation -- Creating Panel Dataset

## Input
- `Data/Data_Collection/Final/Stage_4_Final_w_Calendar_Theme_Removed/model_market_combined_means.parquet` -- Stage 4 combined means table, keyed on `date`
- `Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering/panel_stock_daily_engineered.parquet` -- Panel A, stock-level daily engineered features, keyed on `(permno, date)`
- `Data/Data_Collection/Final/Stage_1_Initial_Merge/panel_macro_daily.parquet` -- Panel C raw macro daily, keyed on `date` (used for target construction only)

## Purpose
Transforms the market-level aggregated time series back into a stock-level panel dataset. Rather than predicting a single market return, this panel formulation allows the model to predict each stock's individual next-day return, using the same market-level features observed on that day augmented with stock-level features for that specific stock. The output is a long-format panel with one row per `(permno, date)` pair, suitable for panel model training.

---

## Pipeline

### Step 1: Load Market-Level Features
The Stage 4 combined means table is loaded. Calendar features are confirmed absent (already removed in Stage 4). Meta columns (`date`, `target_daily_return`) and the monthly target (`target_monthly_return`) are separated from the feature columns. The feature columns form the market-level feature vector that is the same for every stock on a given date.

### Step 2: Load Stock-Level Panel
Panel A is loaded. Only a subset of columns is retained to keep memory manageable: `permno`, `date`, `dlyret`, `dlycap`, and a selected set of stock-level features (returns, volume, bid-ask spread, market cap, beta, and key microstructure features). The stock-level features are winsorised cross-sectionally at 1st/99th percentile per date.

### Step 3: Construct Stock-Level Target
The target for each `(permno, date)` is the stock's **next-day return**: `dlyret` shifted back by 1 within each PERMNO. A date-gap guard nulls out the shifted return where consecutive dates are more than 5 trading days apart.

### Step 4: Filter to Universe
Rows are filtered to only `(permno, date)` pairs where the stock is in the top-100 S&P 500 universe for that year, using the universe annual parquet. Stocks not in universe on a given date are dropped.

### Step 5: Merge Market Features onto Stock Panel
The market-level feature vector from Step 1 is merged onto the stock panel via a left join on `date`. Every stock on a given date receives the same market-level feature vector. Column name conflicts between stock-level and market-level features are resolved with a `mkt_` prefix on the market-level side.

### Step 6: Expanding-Window Z-Standardisation of Stock Features
Stock-level features are z-scored using the same expanding-window approach as Stage 3: `shift(1)` applied to expanding mean and std, minimum 252-day window. This is applied per feature across the full panel (not per stock), so the z-score reflects where each stock sits relative to the cross-sectional and time-series history of that feature.

### Step 7: Drop Warmup Rows and NaN
First 253 rows (by date) are dropped for expanding z-score warmup. Rows with NaN in the target are dropped (last trading day per stock, and gap-boundary rows). Any remaining NaN in features are reported.

### Step 8: Add Cross-Sectional Rank Features
For each of the key stock-level features, the cross-sectional rank (percentile) of each stock within its date cohort is computed and added as an additional feature (`_rank` suffix). Ranks are scaled to [0, 1]. This allows the model to learn from relative positioning (e.g., this stock is in the 90th percentile of turnover today) in addition to the absolute z-scored level.

### Step 9: Validation
- No duplicate `(permno, date)` pairs
- Row count, PERMNO count, date range
- Rows per year with stocks-per-day statistics
- NaN summary across all feature columns
- Target statistics: mean, std, min, max, percentage positive
- **Leakage check:** correlation between target (next-day return) and same-day return must be near zero

### Step 10: Save
Sorted by `(date, permno)` and saved to parquet.

---

## Key Design Decisions
- **Market features are repeated per stock on each date.** Every stock sees the same market context but its own stock-level features. This is the standard panel ML formulation for cross-sectional return prediction.
- **Stock-level z-scoring is panel-wide** (across all stocks and dates), not per-stock. This preserves cross-sectional information -- a stock with unusually high turnover today scores high relative to all stocks on all days, not just its own history.
- **Cross-sectional rank features added** because z-scores can be distorted by outlier stocks; ranks are robust to this.
- **Target is next-day individual stock return**, not the market return. This differs from the aggregated time series models which predict the market-level cap-weighted return.
- **`mkt_` prefix on market-level features** to avoid collision with stock-level features that share base names (e.g., both panels have bid-ask spread, volume, momentum features).

## Output
`Data/Data_Collection/Final/Stage_3_Model_Ready/panel_dataset.parquet` -- keyed on `(permno, date)`, one row per stock-day, containing z-scored stock-level features, market-level features (same per date), cross-sectional rank features, and `target_next_day_ret`

In [5]:
import pandas as pd

# The aggregate model's final feature set -- whatever survived all drops
agg = pd.read_parquet(STAGE4_MEANS_PATH)   # the same Stage 4 means file you already load
agg_cols = set(agg.columns)
print(f"Aggregate final columns: {len(agg_cols)}")

Aggregate final columns: 717


In [2]:
"""
01_creating_panel_dataset.ipynb
================================
Creates the stock-level panel dataset for per-stock crash prediction.

Each row is a (stock, date) observation. One shared set of KAN splines is trained
across all stocks simultaneously, so a z-score of 2 must mean the same thing for
Apple as for Shell, and the same in 2009 as in 2023.

Features:
    Stock-specific: Panel A (daily) + Panel B (monthly), both z-scored with a
      pooled expanding window over all in-universe stock-dates before t. Panel B
      is z-scored at MONTHLY frequency and only then forward-filled to daily, so
      its z does not drift on days when no new data arrived.
    Market-level: Panel C (daily macro) + Panel D (monthly macro), taken from the
      existing aggregate pipeline where they are already z-scored.

Target:
    Per-stock minret_5d = min(dlyret_{t+1} ... dlyret_{t+5})
    y_binary = 1 if minret_5d < -0.02

Output:
    Data/Splits_Panel/
        market_features.parquet
        Split_{A,B,C,D}/panel_{train,val,test}.parquet
        metadata.json, stock_feature_cols.json, market_feature_cols.json
    Data/Diagnostics/panel_preclip/
        panel_preclip_zscores.parquet          full pre-clip frame
        panel_preclip_by_feature.csv           main table, sorted by n_gt5
        panel_preclip_by_stock_feature.csv     per (feature, stock)
        panel_preclip_by_stock.csv             per stock

CHANGES IN THIS VERSION
-----------------------
1. Reads Panel A and Panel B from Stage_3_Normalisation/01_excluded rather than
   Stage 1.5. Stage 1.5 is upstream of the exclusions, so loading from there
   brought back the ISO features and the five discontinued OAP factors -- the
   latter go NaN in late 2023/2024, hit fillna(0.0), and read as "exactly the
   pooled mean" across the whole of Split_D's test period. Fabricated signal in
   a test set.

2. The exclusion list is now also applied to the MARKET features. They come from
   the old Stage 4 file, which still carries rf and vxd_overnight_gap. Filtering
   them through the same list keeps the panel and aggregate pipelines agreeing on
   one set of surviving features.

3. PRE-CLIP DIAGNOSTIC. Phase 5 previously clipped to +/-5 immediately, which is
   the only reason the panel's z-scores have never been measured. The +/-10
   into-estimator cap, the sigma_f ladder and the denominator floor have all
   already run by that point, so saving before the clip gives the direct answer
   to the question that was being deferred: does the cap actually rescue features
   whose scale is dominated by a single observation? std_of_z is the test -- a
   healthy expanding z-score has standard deviation near 1.

4. Phase 2's per-permno merge_asof loop replaced by a single call with
   by='permno'. Identical result, no Python loop over ~230 stocks, and it matches
   the style already used for IBES revenue in 01_prepare_datasets.

5. Phase 6's dead z-scoring block removed. It built three column lists and then
   printed that it was skipping the work.
"""

import json
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

# ── Locate lib/ ──────────────────────────────────────────────────────────────
# __file__ is undefined in Jupyter, so fall back to the working directory. This
# notebook sits in Code/Data_Panel_Creation, so Code/ is one level up.
try:
    _HERE = Path(__file__).resolve().parent
except NameError:
    _HERE = Path.cwd().resolve()
_CODE_ROOT = _HERE.parents[0]
if str(_CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(_CODE_ROOT))

# Purge any cached copy so editing lib/exclusions.py and re-running picks up the
# change without a kernel restart.
for _m in [m for m in list(sys.modules) if m == "lib" or m.startswith("lib.")]:
    del sys.modules[_m]

from lib.exclusions import (  # noqa: E402
    build_column_map, resolve_exclusions, verify_leaks_closed,
)


# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════════

PROJECT_ROOT = Path("../..")

# EDIT 1: post-exclusion inputs, not Stage 1.5.
EXCLUDED_DIR = (PROJECT_ROOT / "Data/Data_Collection/Final"
                / "Stage_3_Normalisation/01_excluded")
PANEL_A_PATH = EXCLUDED_DIR / "panel_stock_daily_engineered.parquet"
PANEL_B_PATH = EXCLUDED_DIR / "panel_stock_monthly_engineered.parquet"

UNIVERSE_PATH = (PROJECT_ROOT / "Data/Data_Collection/Cleaned"
                 / "01_Top100_SP500_Universe/universe_annual_clean.parquet")

# Market features still come from the old Stage 4 file. Transitional: once the
# new Stage 3 normalisation pipeline produces market features, point here at
# those instead. The exclusion list is applied to these columns below so the two
# pipelines do not diverge in the meantime.
STAGE4_MEANS_PATH = (PROJECT_ROOT / "Data/Data_Collection/Final"
                     / "Stage_4_Final_w_Calendar_Theme_Removed"
                     / "model_market_combined_means.parquet")
THEME_CSV_PATH = (PROJECT_ROOT / "Data/Splits/themes"
                  / "combined_means_theme_assignment.csv")
PANEL_C_PATH = (PROJECT_ROOT / "Data/Data_Collection/Final"
                / "Stage_1_5_Validation_and_Feature_Engineering"
                / "panel_macro_daily_engineered.parquet")

OUTPUT_DIR = PROJECT_ROOT / "Data/Splits_Panel"
DIAG_DIR = PROJECT_ROOT / "Data/Diagnostics/panel_preclip"

# Parameters (match 01_prepare_datasets exactly)
TARGET_HORIZON = 5
CRASH_THRESHOLD = -0.02
EMBARGO_DAYS = 5
CLIP_LIMIT = 5.0
DAILY_Z_MIN_DATES = 252
MONTHLY_Z_MIN_DATES = 12
WEEKLY_Z_MIN_PERIODS = 52

# The full pre-clip frame is roughly 525k rows x ~357 features. Useful for
# tracing a specific stock-date, but set False to skip if disk is tight.
SAVE_PRECLIP_PARQUET = True

PANEL_A_ID_COLS = ['permno', 'date']
PANEL_A_TARGET_COL = 'dlyret'
PANEL_A_WEIGHT_COL = 'dlycap'
PANEL_B_ID_COLS = ['permno', 'date']
PANEL_B_WEIGHT_COL = 'month_end_cap'

H41_DELAY_FEATURES = ['fed_assets', 'tga', 'reserves']
WEEKLY_FEATURES = [
    'lev_long', 'lev_short', 'lev_spread', 'am_long', 'am_short', 'am_spread',
    'dealer_long', 'dealer_short', 'dealer_spread', 'other_long', 'other_short',
    'other_spread', 'open_interest', 'lev_net', 'am_net', 'dealer_net',
    'lev_net_pct', 'am_net_pct', 'dealer_net_pct', 'lev_am_ratio',
    'lev_net_chg', 'am_net_chg', 'bullish', 'neutral', 'bearish',
    'bullish_8w_ma', 'bull_bear_spread', 'initial_claims', 'continued_claims',
    'fed_assets', 'tga', 'reserves', 'bank_credit', 'ci_loans',
]

BINARY_FEATURES = [
    'vix_above_20', 'vix_above_30',
    'curve_inverted_2y10y', 'curve_inverted_3m10y', 'credit_stress',
]

META_COLS = {'permno', 'date', 'dlyret', 'dlycap', 'minret_5d', 'y_binary',
             'year'}

SPLITS = {
    "Split_B": {"train_end": "2015-12-31", "val_start": "2016-01-01",
                "val_end": "2017-12-31", "test_start": "2018-01-01",
                "test_end": "2019-12-31"},
    "Split_C": {"train_end": "2017-12-31", "val_start": "2018-01-01",
                "val_end": "2019-12-31", "test_start": "2020-01-01",
                "test_end": "2021-12-31"},
    "Split_A": {"train_end": "2019-12-31", "val_start": "2020-01-01",
                "val_end": "2021-12-31", "test_start": "2022-01-01",
                "test_end": "2023-12-31"},
    "Split_D": {"train_end": "2020-12-31", "val_start": "2021-01-01",
                "val_end": "2022-12-31", "test_start": "2023-01-01",
                "test_end": "2024-12-31"},
}


# ═══════════════════════════════════════════════════════════════════════════════
# PANEL-LEVEL EXPANDING Z-SCORE
# ═══════════════════════════════════════════════════════════════════════════════

def robust_expanding_zscore(df, feature_cols, date_col='date', min_dates=252,
                            sigma_recompute_every=None):
    """
    Robust pooled expanding z-score (pooled across stocks, causal in time).

    TWO DISTINCT REGIMES -- do not conflate them.

    ---------------------------------------------------------------------------
    WARM-UP REGIME: batch, two-pass, computed over the whole block in one go
    ---------------------------------------------------------------------------
    The running regime needs a trustworthy mean and standard deviation to start
    from. Computing them directly from the warm-up block would poison them with
    exactly the extreme values (e.g. ChInvIA ~ -8.7e12) they must be robust to.
    So the warm-up is cleaned first, in three steps done over the entire block at
    once (not row by row):

      1. Z-score every warm-up observation using a ROBUST centre and scale:
             centre = median(x)   -- a quantile, so extremes cannot move it
             scale  = sigma_f     -- scaled MAD, likewise uncorruptible
         The median need not equal the true mean. Its only job is to be an
         uncorruptible anchor for the trim in step 2; any offset it carries is
         corrected in step 3, which recomputes the real mean over cleaned data.

      2. Trim those z-scores to +/-10 and reverse-engineer the raw values:
             x_clean = median + clip(z, -10, +10) * sigma_f
         Equivalently, clip x into [median - 10*sigma_f, median + 10*sigma_f].

      3. Compute the ORDINARY sample mean and sample std over the CLEANED block.
         These seed the running regime. Welford is seeded directly from this
         block -- n, mean, and M2 = var_sample * (n - 1) -- NOT restarted empty,
         so the first post-warm-up observation is z-scored against a fully built
         ruler.

    No z-scores are emitted for warm-up dates; the block exists only to build
    these statistics.

    ---------------------------------------------------------------------------
    RUNNING REGIME: streaming, Welford, everything after warm-up
    ---------------------------------------------------------------------------
      +/-10 into-estimator cap: before a value is accumulated into the running
          moments, its contribution is clipped to mean +/- 10 * expanding_std.
          Real turmoil is allowed to widen the ruler -- which is what makes a
          resulting z of 5 meaningfully rare -- while a glitch cannot dominate
          it. The observation itself still reaches the model uncapped here.
      +/-5 output cap: NOT applied here. Phase 5 clips the final z, giving a
          fixed bounded spline grid identical across features, seeds and
          datasets.

    Denominator floor: max(expanding_std, 0.1 * sigma_f, 1e-8), so a feature
    whose expanding std temporarily collapses cannot turn a trivial move into a
    spurious extreme. sigma_f is refreshed roughly annually from RAW data over
    all history to that point.

      Why raw and not z-scored: sigma_f answers "what is this feature's natural
      spread, in its own units?" -- the only question for which 0.1 * sigma_f is
      a sensible floor. Computed from z-scores it would return ~0.674 (the MAD of
      a standard normal) for every feature regardless of its real scale, making
      the floor a meaningless near-constant, and would be circular.

    Welford rather than sum-of-squares: E[x^2] - E[x]^2 suffers catastrophic
    cancellation on large-magnitude features and can return negative variances.
    Welford is the same O(1) update, numerically stable, and supplies the running
    mean the cap needs as its centre.
    """
    if sigma_recompute_every is None:
        sigma_recompute_every = min_dates

    # kind='stable' so rows within a date keep their input order, making the
    # returned row order deterministic. Callers must still join on keys rather
    # than assigning columns positionally -- see the merge in Phase 4.
    df = df.sort_values(date_col, kind='stable').reset_index(drop=True).copy()

    for col in feature_cols:
        if hasattr(df[col].dtype, 'numpy_dtype'):
            df[col] = df[col].astype('float64')

    unique_dates = np.sort(df[date_col].unique())
    n_dates = len(unique_dates)
    n_features = len(feature_cols)
    n_rows = len(df)

    if n_dates <= min_dates:
        raise ValueError(
            f"Only {n_dates} dates but min_dates={min_dates}: no data would "
            f"survive the warm-up.")

    print(f"    Robust expanding z-score: {n_features} features, "
          f"{n_dates} dates, {n_rows:,} rows")
    print(f"    Warm-up: {min_dates} dates (batch)  |  "
          f"sigma_f refresh: every {sigma_recompute_every} dates")

    raw_matrix = df[feature_cols].values.astype(np.float64)
    z_matrix = np.full((n_rows, n_features), np.nan, dtype=np.float64)

    # Row boundaries per date. Treating each date as a contiguous [start, end)
    # slice is only valid because df is date-sorted; assert rather than assume,
    # since a non-contiguous date would make the slice swallow other dates' rows.
    #
    # The stable sort above is load-bearing in THREE places: raw_matrix, these
    # slices (hence z_matrix), and the r_start indexing used by the annual
    # sigma_f refresh. All three derive from this one sorted frame.
    date_codes = pd.factorize(df[date_col], sort=True)[0]
    date_row_slices = [None] * n_dates
    for d_idx in range(n_dates):
        rows = np.where(date_codes == d_idx)[0]
        assert len(rows) == rows[-1] - rows[0] + 1, (
            f"Date at index {d_idx} occupies non-contiguous rows; the "
            f"[start, end) slicing below would be wrong.")
        date_row_slices[d_idx] = (rows[0], rows[-1] + 1)

    # ══════════════════════════════════════════════════════════════════════
    # Robust scale helper (raw units, quantile-only so extremes cannot move it)
    # ══════════════════════════════════════════════════════════════════════
    def _robust_scale(end_row, report=False):
        """
        Per-feature robust scale over raw_matrix[:end_row]. Floored at 1e-8.

        Every rung is a quantile, so no rung can be shifted by an extreme value:
        a -8.7e12 datum is merely "the smallest value", and half the data would
        have to be corrupted to move a median. Neither an ordinary std nor a mean
        absolute deviation can be used -- both are destroyed by the very values
        sigma_f exists to bound.

          rung 1  Scaled MAD, 1.4826 * median(|x - med|). The normal case.
          rung 2  MAD is 0 (>50% of values identical, so the median absolute
                  deviation vanishes) -> P95(|x - med|) / 1.9600, where 1.9600 is
                  P95 of |N(0,1)| so the result still reads as a standard
                  deviation. Handles ties up to ~95%.
          rung 3  P95 also 0 (>95% ties) -> MAD of the strictly POSITIVE
                  deviations, i.e. spread measured using only the observations
                  that moved. Fails SAFE: a 99%-constant feature with a few
                  enormous movers gets a LARGE scale, so the floor dominates and
                  the feature contributes z ~ 0 -- correct for something that
                  barely moves. Falling through to epsilon would fail DANGEROUS.
          rung 4  Genuinely constant -> epsilon, a numerical catch only.

        Rungs 2 and 3 are load-bearing, not cosmetic. Without them a zero-
        inflated feature gets sigma_f = epsilon, the warm-up trim band collapses
        to [median, median], every genuine non-zero value is flattened onto the
        median, the seeded std is ~0, and post-warm-up z-scores explode until the
        expanding std catches up. Roughly 15 features in the monthly panel need
        rung 2 (ConvDebt at 89.1% ties, ShareRepurchase at 83.8%, and so on).

        For rung-2/3 features sigma_f is a rough proxy scale, not a calibrated
        standard deviation -- a zero-inflated distribution is nowhere near
        Gaussian, so the 1.9600 equivalence does not really hold. Acceptable,
        since sigma_f only feeds the floor and the warm-up trim.
        """
        block = raw_matrix[:end_row]
        out = np.full(n_features, 1e-8)
        rung_used = np.zeros(n_features, dtype=int)
        for j in range(n_features):
            vals = block[:, j]
            valid = vals[~np.isnan(vals)]
            if len(valid) < 2:
                rung_used[j] = 4
                continue
            abs_dev = np.abs(valid - np.median(valid))
            scale = 1.4826 * np.median(abs_dev)                   # rung 1
            rung = 1
            if scale <= 1e-12:
                scale = np.percentile(abs_dev, 95) / 1.9600       # rung 2
                rung = 2
            if scale <= 1e-12:
                pos = abs_dev[abs_dev > 1e-12]                    # rung 3
                scale = 1.4826 * np.median(pos) if len(pos) else 0.0
                rung = 3
            if scale <= 1e-12:
                rung = 4
            out[j] = max(scale, 1e-8)
            rung_used[j] = rung

        if report:
            for r in (2, 3, 4):
                idx = np.where(rung_used == r)[0]
                if len(idx):
                    names = [feature_cols[i] for i in idx[:6]]
                    label = {2: 'rung 2 (P95, >50% ties)',
                             3: 'rung 3 (MAD of movers, >95% ties)',
                             4: 'rung 4 (epsilon, constant)'}[r]
                    print(f"      sigma_f {label}: {len(idx)} feature(s) "
                          f"-> {', '.join(names)}"
                          + (" ..." if len(idx) > 6 else ""))
        return out

    # ══════════════════════════════════════════════════════════════════════
    # WARM-UP REGIME
    # ══════════════════════════════════════════════════════════════════════
    warmup_end_row = date_row_slices[min_dates - 1][1]
    warmup_raw = raw_matrix[:warmup_end_row]

    with np.errstate(all='ignore'):
        warmup_median = np.nanmedian(warmup_raw, axis=0)
    warmup_median = np.where(np.isnan(warmup_median), 0.0, warmup_median)
    sigma_f = _robust_scale(warmup_end_row, report=True)

    lo = warmup_median - 10.0 * sigma_f
    hi = warmup_median + 10.0 * sigma_f
    warmup_clean = np.clip(warmup_raw, lo[np.newaxis, :], hi[np.newaxis, :])
    warmup_clean = np.where(np.isnan(warmup_raw), np.nan, warmup_clean)

    outside_band = (
        ((warmup_raw < lo[np.newaxis, :]) | (warmup_raw > hi[np.newaxis, :]))
        & ~np.isnan(warmup_raw))
    n_trimmed = int(outside_band.sum())

    with np.errstate(all='ignore'):
        w_n = (~np.isnan(warmup_clean)).sum(axis=0).astype(np.float64)
        w_mean = np.nanmean(warmup_clean, axis=0)
        warmup_var = np.nanvar(warmup_clean, axis=0, ddof=1)
    w_mean = np.where(np.isnan(w_mean), 0.0, w_mean)
    warmup_var = np.where(np.isnan(warmup_var), 0.0, warmup_var)
    w_M2 = np.where(w_n > 1, warmup_var * (w_n - 1.0), 0.0)

    floor_f = 0.1 * sigma_f

    n_eps = int((sigma_f <= 1e-8 + 1e-12).sum())
    print(f"    Warm-up block: {warmup_end_row:,} rows")
    print(f"      sigma_f (raw units): "
          f"[{sigma_f.min():.10f}, {sigma_f.max():.4f}]"
          + (f"   ({n_eps} at epsilon)" if n_eps else ""))
    print(f"      trimmed at +/-10 robust sigma: {n_trimmed:,} values")
    print(f"      seeded mean: [{w_mean.min():.4f}, {w_mean.max():.4f}]")
    seed_std = np.sqrt(np.maximum(warmup_var, 0.0))
    print(f"      seeded std:  [{seed_std.min():.6f}, {seed_std.max():.4f}]")

    # ══════════════════════════════════════════════════════════════════════
    # RUNNING REGIME
    # ══════════════════════════════════════════════════════════════════════
    total_capped = 0
    n_sigma_refresh = 0
    last_refresh_idx = min_dates

    for d_idx in range(min_dates, n_dates):
        r_start, r_end = date_row_slices[d_idx]
        raw_vals = raw_matrix[r_start:r_end]
        nan_mask = np.isnan(raw_vals)

        if (d_idx - last_refresh_idx) >= sigma_recompute_every:
            sigma_f = _robust_scale(r_start)
            floor_f = 0.1 * sigma_f
            last_refresh_idx = d_idx
            n_sigma_refresh += 1

        # Expanding std read BEFORE this date's update -- strictly causal.
        expanding_var = np.where(w_n > 1, w_M2 / np.maximum(w_n - 1.0, 1.0), 0.0)
        expanding_std = np.sqrt(np.maximum(expanding_var, 0.0))
        denom = np.maximum(np.maximum(expanding_std, floor_f), 1e-8)

        # OUTPUT
        z_vals = (raw_vals - w_mean[np.newaxis, :]) / denom[np.newaxis, :]
        z_vals[nan_mask] = np.nan
        z_matrix[r_start:r_end] = z_vals

        # STATS UPDATE, contribution capped at +/-10 sigma
        deviations = raw_vals - w_mean[np.newaxis, :]
        prov_abs_z = np.where(nan_mask, 0.0,
                              np.abs(deviations) / denom[np.newaxis, :])
        beyond = (prov_abs_z > 10.0) & (~nan_mask)
        total_capped += int(beyond.sum())

        capped_vals = np.where(
            beyond,
            w_mean[np.newaxis, :]
            + np.sign(deviations) * 10.0 * denom[np.newaxis, :],
            raw_vals)
        capped_vals = np.where(nan_mask, np.nan, capped_vals)

        # Welford parallel merge, one per date. Feature NaN patterns differ, so
        # batch_n must be per-feature and never a shared scalar stock count.
        batch_n = (~nan_mask).sum(axis=0).astype(np.float64)
        has_data = batch_n > 0

        batch_sum = np.nansum(capped_vals, axis=0)
        batch_mean = np.where(has_data, batch_sum / np.maximum(batch_n, 1.0), 0.0)
        batch_devs = np.where(nan_mask, 0.0,
                              capped_vals - batch_mean[np.newaxis, :])
        batch_M2 = np.sum(batch_devs ** 2, axis=0)

        combined_n = w_n + batch_n
        safe_comb = np.where(combined_n > 0, combined_n, 1.0)
        delta = np.where(has_data, batch_mean - w_mean, 0.0)

        w_mean = np.where(has_data, w_mean + delta * batch_n / safe_comb, w_mean)
        w_M2 = np.where(has_data,
                        w_M2 + batch_M2 + delta ** 2 * w_n * batch_n / safe_comb,
                        w_M2)
        w_n = np.where(has_data, combined_n, w_n)

    # ══════════════════════════════════════════════════════════════════════
    # Write back and report
    # ══════════════════════════════════════════════════════════════════════
    # Positional assignment is correct ONLY because z_matrix was built in the row
    # order of this same stably-sorted df. Assert the key column is untouched so
    # the invariant is checked rather than merely reasoned about.
    _keys_before = df[[date_col]].copy()
    df[feature_cols] = z_matrix
    assert df[date_col].equals(_keys_before[date_col]), \
        "Key column changed during z-score write-back; row order is not stable."

    valid = ~np.isnan(z_matrix)
    if valid.any():
        vv = z_matrix[valid]
        print(f"    Post z-score range: [{vv.min():.2f}, {vv.max():.2f}]")
        print(f"    Mean: {vv.mean():.4f}, Std: {vv.std():.4f}")
        print(f"    Valid z-scores: {valid.sum():,} / {z_matrix.size:,} "
              f"({valid.sum() / z_matrix.size * 100:.1f}%)")
        print(f"    Beyond +/-5 (pre-output-clip): "
              f"{int((np.abs(vv) > 5).sum()):,} "
              f"({(np.abs(vv) > 5).mean() * 100:.3f}%)")
        with np.errstate(all='ignore'):
            per_feat_std = np.nanstd(z_matrix, axis=0)
        pf = per_feat_std[~np.isnan(per_feat_std)]
        if len(pf):
            print(f"    Per-feature std: mean={pf.mean():.3f}, "
                  f"min={pf.min():.3f}, max={pf.max():.3f}  "
                  f"(>2: {int((pf > 2).sum())}, <0.5: {int((pf < 0.5).sum())})")
    print(f"    Contributions capped at +/-10 sigma: {total_capped:,}")
    print(f"    sigma_f refreshes: {n_sigma_refresh}")

    return df


# ═══════════════════════════════════════════════════════════════════════════════
# WEEKLY Z-SCORE FIX (for market features, same as 01_prepare_datasets)
# ═══════════════════════════════════════════════════════════════════════════════

def build_weekly_zscored_daily(panel_c_path, weekly_features, h41_delay,
                               min_periods=WEEKLY_Z_MIN_PERIODS):
    """
    Z-score the weekly features at WEEKLY frequency, then forward-fill to daily.

    The original pipeline forward-filled to daily FIRST and z-scored second, so a
    constant numerator over a moving expanding mean/std produced a new z every
    day even when no new data had arrived.
    """
    cols_needed = ['date'] + weekly_features
    panel_c = pd.read_parquet(panel_c_path, columns=cols_needed)
    panel_c['date'] = pd.to_datetime(panel_c['date'])
    panel_c = panel_c.sort_values('date').reset_index(drop=True)
    n_days = len(panel_c)

    result = pd.DataFrame({'date': panel_c['date']})

    for col_name in weekly_features:
        raw = panel_c[col_name].values.copy().astype(np.float64)

        is_update = np.zeros(n_days, dtype=bool)
        is_update[0] = True
        is_update[1:] = np.abs(np.diff(raw)) > 1e-12

        update_idx = np.where(is_update)[0]
        update_vals = raw[update_idx]
        n_updates = len(update_vals)

        z_at_updates = np.full(n_updates, np.nan)
        for k in range(min_periods, n_updates):
            past = update_vals[:k]
            mu = np.mean(past)
            sigma = np.std(past, ddof=1)
            if sigma > 1e-12:
                z_at_updates[k] = (update_vals[k] - mu) / sigma

        daily_z = np.full(n_days, np.nan)
        for k in range(n_updates):
            start = update_idx[k]
            end = update_idx[k + 1] if k + 1 < n_updates else n_days
            daily_z[start:end] = z_at_updates[k]

        # H.4.1 and claims publish Thursday; H.8 publishes Friday of the
        # following week. Shifting these forward one day aligns everything in
        # subtheme 9_7 onto the same day.
        if col_name in h41_delay:
            daily_z = np.roll(daily_z, 1)
            daily_z[0] = np.nan

        result[col_name] = daily_z

    return result


# ═══════════════════════════════════════════════════════════════════════════════
# PRE-CLIP DIAGNOSTIC
# ═══════════════════════════════════════════════════════════════════════════════

def save_preclip_diagnostics(panel, feature_cols, out_dir,
                             panel_a_cols=None, panel_b_cols=None,
                             thresholds=(3.0, 5.0, 10.0, 50.0),
                             clip_limit=CLIP_LIMIT, save_full_parquet=True):
    """
    Measure and save the panel z-scores BEFORE clipping.

    Called from Phase 5 after the bounded forward-fill and before .clip(), so the
    values measured are exactly what the model receives apart from the clip.

    The +/-10 into-estimator cap, the sigma_f ladder and the denominator floor
    have all already run at this point, which makes this the direct test of the
    question that has been deferred throughout: does the cap actually rescue
    features whose scale was dominated by a single observation? std_of_z is the
    answer -- a healthy expanding z-score has standard deviation near 1. Far
    below 1 means the denominator is still inflated and the feature is being
    crushed.

    NaN cells are excluded from every count, which automatically removes the
    warm-up rows (robust_expanding_zscore emits NaN for the first min_dates
    dates) and any stock-months beyond the merge_asof tolerance. It also means
    the fillna(0.0) that happens after the clip cannot contaminate the
    denominators.

    Returns the by-feature table.
    """
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n{'=' * 70}")
    print("PRE-CLIP Z-SCORE DIAGNOSTIC")
    print(f"{'=' * 70}")
    print(f"  After the +/-10 cap, sigma_f ladder and denominator floor;")
    print(f"  BEFORE the +/-{clip_limit} output clip. NaN excluded, so warm-up")
    print(f"  rows drop out automatically.")

    feature_cols = [c for c in feature_cols if c in panel.columns]
    print(f"\n  Rows: {len(panel):,}   Features: {len(feature_cols):,}")
    print(f"  Dates: {panel['date'].min().date()} -> {panel['date'].max().date()}")
    print(f"  Stocks: {panel['permno'].nunique()}")

    if save_full_parquet:
        keep = (['permno', 'date']
                + [c for c in ('dlyret', 'dlycap') if c in panel.columns]
                + feature_cols)
        p = out_dir / "panel_preclip_zscores.parquet"
        panel[keep].to_parquet(p, index=False, engine='pyarrow')
        print(f"\n  Saved: {p.name}  ({p.stat().st_size / 1e6:.0f} MB)")

    # Integer stock codes once; bincount over these beats a per-feature groupby.
    codes, permnos = pd.factorize(panel['permno'], sort=True)
    n_stocks = len(permnos)
    dates = panel['date'].to_numpy()
    permno_vals = panel['permno'].to_numpy()

    a_set = set(panel_a_cols or [])
    b_set = set(panel_b_cols or [])

    tags = [str(t).replace('.0', '') for t in thresholds]
    feat_rows, sf_rows = [], []
    stock_exc = np.zeros(n_stocks, dtype=np.int64)
    stock_obs = np.zeros(n_stocks, dtype=np.int64)

    print(f"\n  Scanning {len(feature_cols):,} features...")
    for i, col in enumerate(feature_cols):
        z = pd.to_numeric(panel[col], errors='coerce').to_numpy(dtype='float64')
        valid = np.isfinite(z)
        n_obs = int(valid.sum())
        if n_obs == 0:
            continue
        az = np.abs(z)
        zv = z[valid]

        block = ('panel_A_daily' if col in a_set
                 else 'panel_B_monthly' if col in b_set else 'unknown')

        # Panel B was z-scored at MONTHLY frequency then forward-filled, so each
        # of its z-scores repeats ~21 times in the daily frame. Raw counts are
        # inflated by exactly that factor and are not comparable to the daily
        # features without dedup.
        if block == 'panel_B_monthly':
            chg = np.ones(n_obs, dtype=bool)
            chg[1:] = np.abs(np.diff(zv)) > 1e-12
            n_obs_dedup = int(chg.sum())
        else:
            n_obs_dedup = n_obs

        rec = {
            'feature': col, 'block': block,
            'n_obs': n_obs, 'n_obs_dedup': n_obs_dedup,
            'std_of_z': float(zv.std(ddof=1)) if n_obs > 1 else np.nan,
            'mean_of_z': float(zv.mean()),
            'max_abs_z': float(az[valid].max()),
        }
        for t, tag in zip(thresholds, tags):
            rec[f'n_gt{tag}'] = int((valid & (az > t)).sum())
        rec['pct_gt5'] = rec['n_gt5'] / n_obs

        j = int(np.argmax(np.where(valid, az, -np.inf)))
        rec['date_of_max'] = str(pd.Timestamp(dates[j]).date())
        rec['permno_of_max'] = int(permno_vals[j])

        # Per-stock concentration: is the feature heavy-tailed across the whole
        # cross-section, or is one stock broken and dragging it along?
        exc = valid & (az > clip_limit)
        ps_exc = np.bincount(codes[exc], minlength=n_stocks)
        ps_obs = np.bincount(codes[valid], minlength=n_stocks)
        n_exc = int(ps_exc.sum())

        rec['n_stocks_with_exc'] = int((ps_exc > 0).sum())
        rec['n_stocks_present'] = int((ps_obs > 0).sum())
        rec['top_stock_share'] = (float(ps_exc.max() / n_exc) if n_exc
                                  else np.nan)
        rec['top_stock_permno'] = (int(permnos[int(ps_exc.argmax())]) if n_exc
                                   else pd.NA)

        # Date concentration: many features peaking on one date means a market
        # event or a bad tape day, not per-feature pathology.
        if n_exc:
            vc = pd.Series(dates[exc]).value_counts()
            rec['top_date_share'] = float(vc.iloc[0] / n_exc)
            rec['top_date'] = str(pd.Timestamp(vc.index[0]).date())
        else:
            rec['top_date_share'] = np.nan
            rec['top_date'] = ''

        feat_rows.append(rec)
        stock_exc += ps_exc
        stock_obs += ps_obs

        for k in np.where(ps_exc > 0)[0]:
            sel = valid & (codes == k)
            sf_rows.append({
                'feature': col, 'block': block, 'permno': int(permnos[k]),
                'n_obs': int(ps_obs[k]), 'n_gt5': int(ps_exc[k]),
                'pct_gt5': float(ps_exc[k] / ps_obs[k]) if ps_obs[k] else np.nan,
                'max_abs_z': float(np.abs(z[sel]).max()) if sel.any() else np.nan,
            })

        if (i + 1) % 50 == 0:
            print(f"    {i + 1}/{len(feature_cols)}")

    by_feature = (pd.DataFrame(feat_rows)
                  .sort_values('n_gt5', ascending=False).reset_index(drop=True))
    by_stock_feature = (pd.DataFrame(sf_rows)
                        .sort_values('n_gt5', ascending=False)
                        .reset_index(drop=True))
    by_stock = (pd.DataFrame({'permno': permnos, 'n_obs': stock_obs,
                              'n_gt5': stock_exc})
                .assign(pct_gt5=lambda d: d['n_gt5']
                        / d['n_obs'].replace(0, np.nan))
                .sort_values('n_gt5', ascending=False).reset_index(drop=True))

    if len(by_stock_feature):
        worst = (by_stock_feature.sort_values('n_gt5', ascending=False)
                 .groupby('permno').first()['feature'])
        by_stock['worst_feature'] = by_stock['permno'].map(worst)

    for df_, nm in ((by_feature, 'panel_preclip_by_feature.csv'),
                    (by_stock_feature, 'panel_preclip_by_stock_feature.csv'),
                    (by_stock, 'panel_preclip_by_stock.csv')):
        df_.to_csv(out_dir / nm, index=False)
        print(f"  Saved: {nm}  ({len(df_):,} rows)")

    # ── Report ──────────────────────────────────────────────────────────────
    tot_obs = int(by_feature['n_obs'].sum())
    tot_exc = int(by_feature['n_gt5'].sum())
    print(f"\n  BOUNDARY MASS (|z| > {clip_limit})")
    print(f"    {tot_exc:,} / {tot_obs:,} = {tot_exc / tot_obs * 100:.4f}%")
    print(f"    Aggregate, for comparison: 0.3066%")
    for t, tag in zip(thresholds, tags):
        e = int(by_feature[f'n_gt{tag}'].sum())
        print(f"      |z| > {t:>5}: {e:>10,} = {e / tot_obs * 100:7.4f}%")

    print(f"\n  BY BLOCK  (Panel B repeats ~21x from the forward-fill, so")
    print(f"  massDedup is its honest figure)")
    print(f"    {'block':<18} {'feats':>6} {'mass%':>8} {'massDedup%':>11} "
          f"{'maxZ':>10}")
    print(f"    {'-' * 58}")
    for blk, g in by_feature.groupby('block'):
        m = g['n_gt5'].sum() / g['n_obs'].sum() * 100
        md = g['n_gt5'].sum() / g['n_obs_dedup'].sum() * 100
        print(f"    {blk:<18} {len(g):>6} {m:>8.4f} {md:>11.4f} "
              f"{g['max_abs_z'].max():>10.1f}")

    print(f"\n  DID THE CAP WORK?  std_of_z should be close to 1.")
    sd = by_feature['std_of_z']
    for lab, m in (("std < 0.10  (crushed)", sd < 0.10),
                   ("0.10-0.50   (suppressed)", (sd >= 0.10) & (sd < 0.50)),
                   ("0.50-1.50   (healthy)", (sd >= 0.50) & (sd < 1.50)),
                   ("1.50-3.00   (loose)", (sd >= 1.50) & (sd < 3.00)),
                   ("std > 3.00  (unstable)", sd >= 3.00)):
        print(f"    {lab:<28} {int(m.sum()):>5} features")

    crushed = by_feature[by_feature['std_of_z'] < 0.50]
    if len(crushed):
        print(f"\n    Suppressed or crushed ({len(crushed)}) -- the cap did not "
              f"fix these:")
        print(f"    {'feature':<40} {'std_of_z':>9} {'maxZ':>10} {'n>5':>6}")
        for _, r in crushed.nsmallest(min(15, len(crushed)),
                                      'std_of_z').iterrows():
            print(f"    {str(r['feature'])[:39]:<40} {r['std_of_z']:>9.5f} "
                  f"{r['max_abs_z']:>10.1f} {int(r['n_gt5']):>6}")

    print(f"\n  TOP 30 BY EXCEEDANCE COUNT")
    print(f"  top_stock_share near 1.0 means one stock is broken; low means the")
    print(f"  whole cross-section is heavy-tailed.")
    print(f"  {'feature':<38} {'n>5':>7} {'pct':>6} {'maxZ':>9} {'stkShr':>7} "
          f"{'nStk':>5} {'stdZ':>6}")
    print(f"  {'-' * 84}")
    for _, r in by_feature.head(30).iterrows():
        ts = ("n/a" if pd.isna(r['top_stock_share'])
              else f"{r['top_stock_share']:.2f}")
        print(f"  {str(r['feature'])[:37]:<38} {int(r['n_gt5']):>7,} "
              f"{r['pct_gt5'] * 100:>5.2f}% {r['max_abs_z']:>9.1f} {ts:>7} "
              f"{int(r['n_stocks_with_exc']):>5} {r['std_of_z']:>6.2f}")

    print(f"\n  TOP 20 BY MAX |z|")
    print(f"  {'feature':<38} {'maxZ':>11} {'date':>12} {'permno':>8} {'stdZ':>6}")
    print(f"  {'-' * 78}")
    for _, r in by_feature.nlargest(20, 'max_abs_z').iterrows():
        print(f"  {str(r['feature'])[:37]:<38} {r['max_abs_z']:>11.1f} "
              f"{r['date_of_max']:>12} {int(r['permno_of_max']):>8} "
              f"{r['std_of_z']:>6.2f}")

    print(f"\n  WORST 20 STOCKS  (a PERMNO here across many unrelated features")
    print(f"  is itself the problem)")
    print(f"  {'permno':>8} {'n_obs':>9} {'n>5':>8} {'pct':>7}  worst feature")
    print(f"  {'-' * 66}")
    for _, r in by_stock.head(20).iterrows():
        wf = r.get('worst_feature', '')
        print(f"  {int(r['permno']):>8} {int(r['n_obs']):>9,} "
              f"{int(r['n_gt5']):>8,} {r['pct_gt5'] * 100:>6.2f}%  "
              f"{'' if pd.isna(wf) else str(wf)[:34]}")

    if len(by_stock) > 1 and tot_exc:
        share = by_stock['n_gt5'].head(5).sum() / tot_exc * 100
        print(f"\n    Top 5 stocks hold {share:.1f}% of all exceedances "
              f"({5 / len(by_stock) * 100:.1f}% of stocks).")

    print(f"\n  DATES RECURRING AS A FEATURE MAXIMUM")
    vc = by_feature[by_feature['max_abs_z'] > 10]['date_of_max'].value_counts()
    for d, n in vc.head(12).items():
        print(f"    {d}   {n:>4} features")

    print(f"\n  NEXT")
    print(f"    1. Sort panel_preclip_by_feature.csv by n_gt5 and read the top.")
    print(f"    2. Anything with top_stock_share above ~0.5: look it up in")
    print(f"       panel_preclip_by_stock_feature.csv. One stock is likely")
    print(f"       responsible and the fix is that stock, not the feature.")
    print(f"    3. Check std_of_z for every deferred-register entry. Near 1")
    print(f"       means the cap worked and the entry closes as 'keep'.")
    print(f"    4. Use panel_preclip_zscores.parquet to pull the offending")
    print(f"       stock-date for anything still unexplained.")

    return by_feature


# ═══════════════════════════════════════════════════════════════════════════════
# MAIN PIPELINE
# ═══════════════════════════════════════════════════════════════════════════════

def main():
    print("=" * 70)
    print("PANEL DATASET CREATION")
    print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 70)
    print(f"  Inputs: {EXCLUDED_DIR}")

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # ══════════════════════════════════════════════════════════════════════
    # PHASE 1: LOAD AND FILTER TO IN-UNIVERSE
    # ══════════════════════════════════════════════════════════════════════
    print(f"\n{'=' * 70}")
    print("PHASE 1: LOAD AND FILTER TO IN-UNIVERSE")
    print(f"{'=' * 70}")

    universe = pd.read_parquet(UNIVERSE_PATH)
    universe_pairs = set(zip(universe['permno'], universe['year']))
    print(f"  Universe: {len(universe)} (permno, year) pairs, "
          f"{universe['permno'].nunique()} unique PERMNOs, "
          f"{universe['year'].nunique()} years")

    # ── Panel A ─────────────────────────────────────────────────────────────
    print(f"\n  Loading Panel A (stock daily, post-exclusion)...")
    if not PANEL_A_PATH.exists():
        raise FileNotFoundError(
            f"{PANEL_A_PATH} not found. Run "
            f"Data_Merging/Stage_3_Normalisation/01_apply_exclusions first.")
    panel_a = pd.read_parquet(PANEL_A_PATH)
    panel_a['date'] = pd.to_datetime(panel_a['date'])
    # Force plain numpy int64. The two parquet files can disagree -- one carries
    # nullable Int64Dtype and the other numpy int64 -- and merge_asof (unlike
    # merge) refuses to join keys of different dtypes:
    #   MergeError: incompatible merge keys [0] Int64Dtype() and dtype('int64')
    # The old per-permno loop hid this by reassigning merged['permno'] = permno
    # with a Python int on every iteration. Normalising at load is the honest fix
    # and also keeps the (permno, date) dict lookup in Phase 4 well behaved.
    panel_a['permno'] = panel_a['permno'].astype('int64')
    panel_a['year'] = panel_a['date'].dt.year
    n_before = len(panel_a)

    # Vectorised membership test on the (permno, year) pair. A row-wise .apply
    # costs minutes on ~500k rows; a MultiIndex .isin is near-instant.
    _pair_idx = pd.MultiIndex.from_arrays(
        [panel_a['permno'].values, panel_a['year'].values])
    panel_a = panel_a[_pair_idx.isin(universe_pairs)].reset_index(drop=True)
    print(f"    Before filter: {n_before:,} rows")
    print(f"    After filter:  {len(panel_a):,} rows")
    print(f"    PERMNOs: {panel_a['permno'].nunique()}")
    print(f"    Dates: {panel_a['date'].min().date()} -> "
          f"{panel_a['date'].max().date()}")
    print(f"    Stocks/day: "
          f"mean={panel_a.groupby('date')['permno'].nunique().mean():.1f}, "
          f"min={panel_a.groupby('date')['permno'].nunique().min()}")

    panel_a_feat_cols = [
        c for c in panel_a.columns
        if c not in META_COLS
        and c not in {PANEL_A_TARGET_COL, PANEL_A_WEIGHT_COL}
        and c not in PANEL_A_ID_COLS]
    print(f"    Factor columns: {len(panel_a_feat_cols)}")

    assert panel_a.duplicated(subset=['permno', 'date']).sum() == 0, \
        "Duplicate (permno, date) in Panel A"
    assert panel_a[PANEL_A_TARGET_COL].notna().all(), "NaN in dlyret"
    assert panel_a[PANEL_A_WEIGHT_COL].notna().all(), "NaN in dlycap"
    print(f"    No duplicate (permno, date), no NaN in target/weight")

    # ── Panel B ─────────────────────────────────────────────────────────────
    print(f"\n  Loading Panel B (stock monthly, post-exclusion)...")
    if not PANEL_B_PATH.exists():
        raise FileNotFoundError(
            f"{PANEL_B_PATH} not found. Run 01_apply_exclusions first.")
    panel_b = pd.read_parquet(PANEL_B_PATH)
    panel_b['date'] = pd.to_datetime(panel_b['date'])
    panel_b['permno'] = panel_b['permno'].astype('int64')   # see Panel A above
    panel_b['year'] = panel_b['date'].dt.year
    n_before = len(panel_b)
    _pair_idx = pd.MultiIndex.from_arrays(
        [panel_b['permno'].values, panel_b['year'].values])
    panel_b = panel_b[_pair_idx.isin(universe_pairs)].reset_index(drop=True)
    print(f"    Before filter: {n_before:,} rows")
    print(f"    After filter:  {len(panel_b):,} rows")
    print(f"    PERMNOs: {panel_b['permno'].nunique()}")
    print(f"    Dates: {panel_b['date'].min().date()} -> "
          f"{panel_b['date'].max().date()}")

    panel_b_non_feat = (META_COLS | set(PANEL_B_ID_COLS)
                        | {PANEL_B_WEIGHT_COL, 'month_end_price'})
    panel_b_feat_cols = [c for c in panel_b.columns
                         if c not in panel_b_non_feat]
    print(f"    Factor columns: {len(panel_b_feat_cols)}")

    assert panel_b.duplicated(subset=['permno', 'date']).sum() == 0, \
        "Duplicate (permno, date) in Panel B"
    print(f"    No duplicate (permno, date)")

    # ── Confirm the exclusions actually landed ──────────────────────────────
    # ExchSwitch, the ISO features and the five discontinued OAP factors are all
    # handled by lib/exclusions.py now. The ad-hoc ExchSwitch drop that used to
    # live here is gone; this asserts the upstream step did its job instead.
    for nm, cols in (('Panel A', panel_a_feat_cols),
                     ('Panel B', panel_b_feat_cols)):
        leaks = {g: v for g, v in verify_leaks_closed(cols).items() if v}
        assert not leaks, (
            f"{nm} still carries known leak factors {leaks}. The inputs are not "
            f"the post-exclusion files.")
    assert 'ExchSwitch' not in panel_b_feat_cols, \
        "ExchSwitch present -- inputs are not post-exclusion"
    print(f"\n  Exclusions verified: no ISO, no discontinued OAP, no ExchSwitch")

    # ══════════════════════════════════════════════════════════════════════
    # PHASE 2: Z-SCORE PANEL B (MONTHLY) THEN FORWARD-FILL TO DAILY
    # ══════════════════════════════════════════════════════════════════════
    print(f"\n{'=' * 70}")
    print("PHASE 2: Z-SCORE PANEL B (MONTHLY) + FORWARD-FILL TO DAILY")
    print(f"{'=' * 70}")

    print(f"\n  Z-scoring Panel B at monthly frequency...")
    panel_b_z = robust_expanding_zscore(
        panel_b[PANEL_B_ID_COLS + panel_b_feat_cols].copy(),
        panel_b_feat_cols, date_col='date',
        min_dates=MONTHLY_Z_MIN_DATES, sigma_recompute_every=12)

    print(f"\n  Forward-filling Panel B to daily...")

    # Prefix so Panel B features cannot collide with Panel A.
    panel_b_z = panel_b_z.rename(columns={
        c: f'monthly_{c}' for c in panel_b_feat_cols
        if not c.startswith('monthly_')})
    panel_b_feat_cols_daily = [
        c if c.startswith('monthly_') else f'monthly_{c}'
        for c in panel_b_feat_cols]

    # One merge_asof with by='permno' replaces the previous loop over ~230
    # stocks. Identical result: merge_asof is a left join, so a stock with no
    # Panel B rows gets NaN, and the 95-day tolerance bounds staleness to about
    # three months so a delisted stock's last monthly value cannot persist.
    daily_spine = panel_a[['permno', 'date']].sort_values('date')

    # merge_asof requires the `by` keys to be the exact same dtype, so state the
    # requirement rather than trusting that the loads normalised it.
    assert daily_spine['permno'].dtype == panel_b_z['permno'].dtype, (
        f"permno dtype mismatch: spine {daily_spine['permno'].dtype} vs "
        f"panel_b_z {panel_b_z['permno'].dtype}. merge_asof will not join keys "
        f"of different dtypes.")

    panel_b_daily = pd.merge_asof(
        daily_spine,
        panel_b_z.sort_values('date'),
        on='date', by='permno', direction='backward',
        tolerance=pd.Timedelta(days=95))

    assert len(panel_b_daily) == len(panel_a), (
        f"Panel B daily rows ({len(panel_b_daily)}) != Panel A rows "
        f"({len(panel_a)})")

    n_nan_b = panel_b_daily[panel_b_feat_cols_daily].isna().sum().sum()
    n_cells_b = len(panel_b_daily) * len(panel_b_feat_cols_daily)
    print(f"    Panel B daily: {len(panel_b_daily):,} rows, "
          f"{len(panel_b_feat_cols_daily)} features")
    print(f"    NaN: {n_nan_b:,} / {n_cells_b:,} "
          f"({n_nan_b / n_cells_b * 100:.2f}%)")

    # ══════════════════════════════════════════════════════════════════════
    # PHASE 3: Z-SCORE PANEL A (DAILY)
    # ══════════════════════════════════════════════════════════════════════
    print(f"\n{'=' * 70}")
    print("PHASE 3: Z-SCORE PANEL A (DAILY)")
    print(f"{'=' * 70}")

    panel_a_z = robust_expanding_zscore(
        panel_a[PANEL_A_ID_COLS + panel_a_feat_cols].copy(),
        panel_a_feat_cols, date_col='date',
        min_dates=DAILY_Z_MIN_DATES, sigma_recompute_every=252)

    # ══════════════════════════════════════════════════════════════════════
    # PHASE 4: MERGE PANELS + ADD TARGETS
    # ══════════════════════════════════════════════════════════════════════
    print(f"\n{'=' * 70}")
    print("PHASE 4: MERGE PANELS + ADD TARGETS")
    print(f"{'=' * 70}")

    # Attach target and weight by KEY, never positionally.
    #
    # robust_expanding_zscore sorts by date, whereas panel_a arrives permno-major
    # from the partitioned parquet, so the two frames are in different row
    # orders. Assigning panel_a[...].values positionally onto panel_a_z would
    # silently attach each stock-day the WRONG stock's return -- and Phase 7's
    # spot check could not detect it, because it would verify minret_5d against
    # that same misaligned dlyret and find them internally consistent.
    panel = panel_a_z.merge(
        panel_a[['permno', 'date', PANEL_A_TARGET_COL, PANEL_A_WEIGHT_COL]],
        on=['permno', 'date'], how='left', validate='one_to_one')
    assert len(panel) == len(panel_a), "Row count changed on target/weight merge"
    assert panel[PANEL_A_TARGET_COL].notna().all(), "NaN dlyret after merge"
    assert panel[PANEL_A_WEIGHT_COL].notna().all(), "NaN dlycap after merge"

    # Independent alignment check: re-derive dlyret via a dict keyed on
    # (permno, date) and require an exact match. Catches any future reordering.
    _key_to_ret = dict(zip(
        zip(panel_a['permno'].values, panel_a['date'].values),
        panel_a[PANEL_A_TARGET_COL].values))
    _expected_ret = np.array([
        _key_to_ret[(p, d)]
        for p, d in zip(panel['permno'].values, panel['date'].values)])
    _max_ret_diff = np.abs(
        _expected_ret - panel[PANEL_A_TARGET_COL].values).max()
    assert _max_ret_diff < 1e-12, (
        f"Target misalignment: max diff {_max_ret_diff:.3e}. dlyret is not "
        f"correctly keyed to (permno, date).")
    print(f"  Target/weight attached by key "
          f"(alignment verified, max diff {_max_ret_diff:.1e})")

    panel = panel.merge(
        panel_b_daily[['permno', 'date'] + panel_b_feat_cols_daily],
        on=['permno', 'date'], how='left', validate='one_to_one')
    assert len(panel) == len(panel_a), "Row explosion on Panel B merge"

    all_stock_feat_cols = panel_a_feat_cols + panel_b_feat_cols_daily
    print(f"  Merged panel: {len(panel):,} rows, "
          f"{len(all_stock_feat_cols)} stock features")
    print(f"    Panel A: {len(panel_a_feat_cols)}   "
          f"Panel B: {len(panel_b_feat_cols_daily)}")

    # ══════════════════════════════════════════════════════════════════════
    # PHASE 5: DIAGNOSE, CLIP, FILL NaN
    # ══════════════════════════════════════════════════════════════════════
    print(f"\n{'=' * 70}")
    print("PHASE 5: DIAGNOSE, CLIP, FILL NaN")
    print(f"{'=' * 70}")

    # Bounded forward-fill within each stock, Panel A only. Panel B staleness is
    # already bounded to ~3 months by the merge_asof tolerance in Phase 2, and
    # re-filling here would undo that cut-off.
    panel = panel.sort_values(['permno', 'date']).reset_index(drop=True)
    n_nan_before_ffill = panel[panel_a_feat_cols].isna().sum().sum()
    panel[panel_a_feat_cols] = (
        panel.groupby('permno')[panel_a_feat_cols].ffill(limit=5))
    n_nan_after_ffill = panel[panel_a_feat_cols].isna().sum().sum()
    print(f"  Per-stock 5-day forward-fill (Panel A only): "
          f"{n_nan_before_ffill:,} -> {n_nan_after_ffill:,} NaN "
          f"({n_nan_before_ffill - n_nan_after_ffill:,} filled)")

    # EDIT 3: measure and save before the clip removes the tail.
    save_preclip_diagnostics(
        panel, all_stock_feat_cols, DIAG_DIR,
        panel_a_cols=panel_a_feat_cols,
        panel_b_cols=panel_b_feat_cols_daily,
        clip_limit=CLIP_LIMIT,
        save_full_parquet=SAVE_PRECLIP_PARQUET)

    print(f"\n{'=' * 70}")
    print("PHASE 5 (continued): CLIP AND FILL")
    print(f"{'=' * 70}")

    n_beyond = (panel[all_stock_feat_cols].abs() > CLIP_LIMIT).sum().sum()
    panel[all_stock_feat_cols] = panel[all_stock_feat_cols].clip(
        lower=-CLIP_LIMIT, upper=CLIP_LIMIT)
    print(f"  Clipped {n_beyond:,} values to +/-{CLIP_LIMIT}")

    # fillna(0) imputes "exactly the pooled mean". Where a feature has many
    # missings this puts a spike of mass at 0, visible in the plotted spline.
    n_nan_pre = panel[all_stock_feat_cols].isna().sum().sum()
    panel[all_stock_feat_cols] = panel[all_stock_feat_cols].fillna(0.0)
    print(f"  NaN filled with 0: {n_nan_pre:,} "
          f"({n_nan_pre / (len(panel) * len(all_stock_feat_cols)) * 100:.2f}% "
          f"of cells)")
    assert panel[all_stock_feat_cols].isna().sum().sum() == 0, \
        "NaN remaining in stock features"

    # Trim to the market feature dates. That file starts ~mid-2008, after every
    # upstream warm-up (12-month monthly, 252-day daily, 252-day macro), so this
    # single trim guarantees all three are satisfied and that every stock-day has
    # a matching market row.
    print(f"\n  Trimming to market feature dates...")
    market_dates_df = pd.read_parquet(STAGE4_MEANS_PATH, columns=['date'])
    market_dates_df['date'] = pd.to_datetime(market_dates_df['date'])
    valid_dates = set(market_dates_df['date'].values)
    n_before = len(panel)
    panel = panel[panel['date'].isin(valid_dates)].reset_index(drop=True)
    print(f"  {n_before:,} -> {len(panel):,} rows")
    print(f"  Date range: {panel['date'].min().date()} -> "
          f"{panel['date'].max().date()}   "
          f"({panel['date'].nunique()} trading days)")

    # ══════════════════════════════════════════════════════════════════════
    # PHASE 6: EXTRACT MARKET FEATURES
    # ══════════════════════════════════════════════════════════════════════
    print(f"\n{'=' * 70}")
    print("PHASE 6: EXTRACT MARKET FEATURES")
    print(f"{'=' * 70}")

    theme_df = pd.read_csv(THEME_CSV_PATH)
    market_col_names = theme_df[
        theme_df['panel'].str.contains('C|D', na=False)]['column'].tolist()
    print(f"  Panel C/D features named in the theme CSV: {len(market_col_names)}")

    stage4 = pd.read_parquet(STAGE4_MEANS_PATH)
    stage4['date'] = pd.to_datetime(stage4['date'])
    stage4 = stage4.sort_values('date').reset_index(drop=True)

    market_cols_present = [c for c in market_col_names if c in stage4.columns]
    missing = [c for c in market_col_names if c not in stage4.columns]
    if missing:
        print(f"  {len(missing)} named but absent from Stage 4: {missing[:5]}")

    # EDIT 2: the market features come from the OLD Stage 4 file, so they still
    # carry rf and vxd_overnight_gap. Filter them through the same exclusion list
    # the stock panels went through, or the two pipelines disagree about which
    # features exist.
    mkt_colmap = build_column_map(market_cols_present)
    mkt_excl = resolve_exclusions(market_cols_present, mkt_colmap,
                                  for_pipeline='aggregate', stage='now')
    if mkt_excl.n_drop:
        print(f"\n  Applying the exclusion list to market features: "
              f"{mkt_excl.n_drop} dropped")
        for c in mkt_excl.to_drop:
            print(f"    {c}")
        market_cols_present = [c for c in market_cols_present
                               if c not in set(mkt_excl.to_drop)]
    else:
        print(f"\n  Exclusion list drops no market features")

    market_features = stage4[['date'] + market_cols_present].copy()
    print(f"  Market features: {len(market_cols_present)} columns, "
          f"{len(market_features)} dates")

    # Weekly features: replace the daily-drifted Stage 3 z-scores with
    # weekly-frequency ones.
    weekly_in_market = [c for c in WEEKLY_FEATURES
                        if c in market_features.columns]
    if weekly_in_market:
        weekly_z = build_weekly_zscored_daily(
            PANEL_C_PATH, WEEKLY_FEATURES, H41_DELAY_FEATURES)
        market_features = market_features.drop(columns=weekly_in_market)
        market_features = market_features.merge(
            weekly_z[['date'] + weekly_in_market], on='date', how='left')
        print(f"  Replaced {len(weekly_in_market)} weekly features with "
              f"weekly-frequency z-scores")

    # Market features arrive already z-scored from the aggregate pipeline's
    # Stage 3, so they are only clipped and NaN-filled here. (The previous
    # version built three column lists at this point and then printed that it
    # was skipping the work.)
    clip_cols = [c for c in market_cols_present if c not in BINARY_FEATURES]
    n_beyond_m = (market_features[clip_cols].abs() > CLIP_LIMIT).sum().sum()
    market_features[clip_cols] = market_features[clip_cols].clip(
        lower=-CLIP_LIMIT, upper=CLIP_LIMIT)
    n_nan_m = market_features[market_cols_present].isna().sum().sum()
    market_features[market_cols_present] = \
        market_features[market_cols_present].fillna(0.0)
    print(f"  Clipped {n_beyond_m:,} values; filled {n_nan_m:,} NaN")

    panel_dates = set(panel['date'].unique())
    market_features = market_features[
        market_features['date'].isin(panel_dates)].reset_index(drop=True)
    print(f"  Trimmed to panel dates: {len(market_features)} rows")

    # ══════════════════════════════════════════════════════════════════════
    # PHASE 7: COMPUTE PER-STOCK TARGETS
    # ══════════════════════════════════════════════════════════════════════
    print(f"\n{'=' * 70}")
    print("PHASE 7: COMPUTE PER-STOCK TARGETS")
    print(f"{'=' * 70}")

    panel = panel.sort_values(['permno', 'date'],
                             kind='stable').reset_index(drop=True)

    # minret_5d[t] = min(dlyret[t+1] ... dlyret[t+5]), per stock. Vectorised via
    # five per-group shifts. groupby().shift() never reaches across a permno
    # boundary, so there is no cross-stock bleed and the last TARGET_HORIZON rows
    # of every stock correctly become NaN -- which also handles delistings.
    print(f"  Computing per-stock minret_{TARGET_HORIZON}d...")
    _g = panel.groupby('permno', sort=False)[PANEL_A_TARGET_COL]
    _shifted = pd.concat([_g.shift(-k)
                          for k in range(1, TARGET_HORIZON + 1)], axis=1)
    panel['minret_5d'] = _shifted.min(axis=1)
    # min(axis=1) skips NaN, which would silently take the min of a PARTIAL
    # window at each stock's tail. Require the full window.
    panel.loc[_shifted.isna().any(axis=1), 'minret_5d'] = np.nan

    panel['y_binary'] = (panel['minret_5d'] < CRASH_THRESHOLD).astype(float)
    panel.loc[panel['minret_5d'].isna(), 'y_binary'] = np.nan

    n_valid = panel['y_binary'].notna().sum()
    n_crash = int(panel['y_binary'].sum())
    print(f"  Valid targets: {n_valid:,} / {len(panel):,}")
    print(f"  Crash rate: {n_crash / n_valid:.1%} ({n_crash:,} events)")
    print(f"  minret_5d range: [{panel['minret_5d'].min():.4f}, "
          f"{panel['minret_5d'].max():.4f}]")

    # ── External target validation ──────────────────────────────────────────
    # Re-derives the target from the RAW parquet rather than from panel's own
    # dlyret. A check that reads panel['dlyret'] cannot detect a misalignment,
    # because it would verify minret_5d against the same misaligned returns and
    # find them consistent -- which is how the earlier positional-assignment bug
    # went unnoticed.
    print(f"\n  External target validation (independent of panel['dlyret'])...")
    _raw = pd.read_parquet(PANEL_A_PATH,
                           columns=['permno', 'date', PANEL_A_TARGET_COL])
    _raw['date'] = pd.to_datetime(_raw['date'])
    _raw = _raw.sort_values(['permno', 'date'],
                            kind='stable').reset_index(drop=True)

    _rng = np.random.default_rng(0)
    _cand = panel.index[panel['minret_5d'].notna()].values
    _sample_idx = _rng.choice(_cand, size=min(200, len(_cand)), replace=False)

    _raw_by_permno = {p: g for p, g in _raw.groupby('permno', sort=False)}
    _n_checked, _worst = 0, 0.0
    for _i in _sample_idx:
        _p, _d = panel.at[_i, 'permno'], panel.at[_i, 'date']
        _gp = _raw_by_permno.get(_p)
        if _gp is None:
            continue
        _pos = _gp['date'].searchsorted(_d)
        if _pos >= len(_gp) or _gp['date'].iloc[_pos] != _d:
            continue
        _fwd = _gp[PANEL_A_TARGET_COL].values[
            _pos + 1: _pos + 1 + TARGET_HORIZON]
        if len(_fwd) < TARGET_HORIZON or np.isnan(_fwd).any():
            continue
        _worst = max(_worst, abs(np.min(_fwd) - panel.at[_i, 'minret_5d']))
        _n_checked += 1

    assert _n_checked > 50, (
        f"Only {_n_checked} rows could be externally validated -- too few to "
        f"trust; check the (permno, date) keying.")
    assert _worst < 1e-10, (
        f"EXTERNAL target validation FAILED: max diff {_worst:.3e} across "
        f"{_n_checked} sampled rows. minret_5d does not match the raw parquet, "
        f"indicating dlyret is misaligned to (permno, date).")
    print(f"    Verified {_n_checked} random rows against the raw parquet, "
          f"max diff {_worst:.1e}")
    del _raw, _raw_by_permno

    # ══════════════════════════════════════════════════════════════════════
    # PHASE 8: SPLIT, EMBARGO, SAVE
    # ══════════════════════════════════════════════════════════════════════
    print(f"\n{'=' * 70}")
    print("PHASE 8: SPLIT, EMBARGO, SAVE")
    print(f"{'=' * 70}")

    market_path = OUTPUT_DIR / "market_features.parquet"
    market_features.to_parquet(market_path, index=False, engine='pyarrow')
    print(f"\n  Saved market features: {market_features.shape}")

    with open(OUTPUT_DIR / "stock_feature_cols.json", 'w') as f:
        json.dump(all_stock_feat_cols, f, indent=2)
    with open(OUTPUT_DIR / "market_feature_cols.json", 'w') as f:
        json.dump(market_cols_present, f, indent=2)
    print(f"  Saved feature column lists")

    all_metadata = {}
    save_cols = (['permno', 'date', PANEL_A_TARGET_COL, PANEL_A_WEIGHT_COL,
                  'minret_5d', 'y_binary'] + all_stock_feat_cols)

    for split_name, cfg in SPLITS.items():
        print(f"\n  --- {split_name} ---")
        split_dir = OUTPUT_DIR / split_name
        split_dir.mkdir(parents=True, exist_ok=True)

        dates = pd.to_datetime(panel['date'])
        parts = {
            'train': panel[dates <= cfg['train_end']].copy(),
            'val': panel[(dates >= cfg['val_start'])
                         & (dates <= cfg['val_end'])].copy(),
            'test': panel[(dates >= cfg['test_start'])
                          & (dates <= cfg['test_end'])].copy(),
        }

        # Embargo: drop the last EMBARGO_DAYS distinct dates from train and val,
        # matching TARGET_HORIZON so the forward-looking window cannot straddle a
        # boundary.
        for nm in ('train', 'val'):
            ud = sorted(parts[nm]['date'].unique())
            if len(ud) > EMBARGO_DAYS:
                cutoff = ud[-EMBARGO_DAYS]
                parts[nm] = parts[nm][parts[nm]['date'] < cutoff]

        for nm in parts:
            parts[nm] = (parts[nm].dropna(subset=['y_binary'])
                         .reset_index(drop=True))

        for nm, part_df in parts.items():
            part_df[save_cols].to_parquet(
                split_dir / f"panel_{nm}.parquet", index=False,
                engine='pyarrow')

            n_dates = part_df['date'].nunique()
            n_stocks = part_df['permno'].nunique()
            crash_rate = part_df['y_binary'].mean()
            n_crash = int(part_df['y_binary'].sum())
            print(f"    {nm}: {len(part_df):,} rows, {n_dates} dates, "
                  f"{n_stocks} stocks, crash: {crash_rate:.1%} ({n_crash})")

            assert part_df[all_stock_feat_cols].isna().sum().sum() == 0, \
                f"NaN in {nm} stock features"
            assert part_df['y_binary'].isna().sum() == 0, \
                f"NaN in {nm} y_binary"

            all_metadata[f"{split_name}/{nm}"] = {
                'rows': len(part_df), 'dates': n_dates, 'stocks': n_stocks,
                'crash_rate': float(crash_rate), 'crash_count': n_crash,
                'date_range': [str(part_df['date'].min()),
                               str(part_df['date'].max())]}

        td = set(parts['train']['date'].unique())
        vd = set(parts['val']['date'].unique())
        sd = set(parts['test']['date'].unique())
        assert not (td & vd), "Train/val date overlap"
        assert not (vd & sd), "Val/test date overlap"
        assert not (td & sd), "Train/test date overlap"
        print(f"    No date overlap between splits")

    # ══════════════════════════════════════════════════════════════════════
    # PHASE 9: METADATA AND SUMMARY
    # ══════════════════════════════════════════════════════════════════════
    meta = {
        'created': datetime.now().isoformat(),
        'inputs': {'panel_a': str(PANEL_A_PATH), 'panel_b': str(PANEL_B_PATH)},
        'target': f'per-stock minret_{TARGET_HORIZON}d < {CRASH_THRESHOLD}',
        'embargo_days': EMBARGO_DAYS, 'clip_limit': CLIP_LIMIT,
        'daily_z_min_dates': DAILY_Z_MIN_DATES,
        'monthly_z_min_dates': MONTHLY_Z_MIN_DATES,
        'n_stock_features': len(all_stock_feat_cols),
        'n_market_features': len(market_cols_present),
        'n_panel_a_features': len(panel_a_feat_cols),
        'n_panel_b_features': len(panel_b_feat_cols_daily),
        'preclip_diagnostics': str(DIAG_DIR),
        'splits': all_metadata,
    }
    with open(OUTPUT_DIR / "metadata.json", 'w') as f:
        json.dump(meta, f, indent=2, default=str)

    print(f"\n{'=' * 70}")
    print("SUMMARY")
    print(f"{'=' * 70}")
    print(f"\n  Stock features:  {len(all_stock_feat_cols)} "
          f"(A: {len(panel_a_feat_cols)}, B: {len(panel_b_feat_cols_daily)})")
    print(f"  Market features: {len(market_cols_present)}")
    print(f"  Clip: +/-{CLIP_LIMIT}   Burn-in: {DAILY_Z_MIN_DATES}d / "
          f"{MONTHLY_Z_MIN_DATES}m")

    print(f"\n  {'Split':<10} {'Part':<6} {'Rows':>8} {'Dates':>6} "
          f"{'Stocks':>7} {'Crash%':>7}")
    print(f"  {'-' * 50}")
    for key in sorted(all_metadata):
        m = all_metadata[key]
        s, p = key.split('/')
        print(f"  {s:<10} {p:<6} {m['rows']:>8,} {m['dates']:>6} "
              f"{m['stocks']:>7} {m['crash_rate']:>6.1%}")

    print(f"\n  Splits:      {OUTPUT_DIR}")
    print(f"  Diagnostics: {DIAG_DIR}")
    print(f"\n{'=' * 70}")
    print(f"Completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'=' * 70}")
    print(f"""
  READ NEXT: {DIAG_DIR / 'panel_preclip_by_feature.csv'}

  Sort by n_gt5. The columns that matter:

    std_of_z          near 1 = healthy. This is the test of whether the +/-10
                      cap worked. Every entry on the deferred register can be
                      closed on this one number.
    top_stock_share   near 1.0 = one stock is broken and dragging the feature
                      with it; low = the whole cross-section is heavy-tailed and
                      the cap is the correct treatment.
    massDedup%        the honest figure for Panel B features, which repeat ~21x
                      from the forward-fill.

  Compare the total boundary mass against the aggregate's 0.3066%.""")


# ═══════════════════════════════════════════════════════════════════════════════
# LOADER UTILITY
# ═══════════════════════════════════════════════════════════════════════════════

def load_panel_split(split_name="Split_A", part="train", base_dir=OUTPUT_DIR):
    """Load a prepared panel split for model training."""
    panel = pd.read_parquet(base_dir / split_name / f"panel_{part}.parquet")
    market = pd.read_parquet(base_dir / "market_features.parquet")

    with open(base_dir / "stock_feature_cols.json") as f:
        stock_feat_cols = json.load(f)
    with open(base_dir / "market_feature_cols.json") as f:
        market_feat_cols = json.load(f)

    panel_dates = set(panel['date'].unique())
    market = market[market['date'].isin(panel_dates)].reset_index(drop=True)

    return {'panel': panel, 'market': market,
            'stock_feature_cols': stock_feat_cols,
            'market_feature_cols': market_feat_cols}


if __name__ == "__main__":
    main()

PANEL DATASET CREATION
Started: 2026-07-31 10:30:36
  Inputs: ..\..\Data\Data_Collection\Final\Stage_3_Normalisation\01_excluded

PHASE 1: LOAD AND FILTER TO IN-UNIVERSE
  Universe: 2100 (permno, year) pairs, 227 unique PERMNOs, 21 years

  Loading Panel A (stock daily, post-exclusion)...
    Before filter: 525,957 rows
    After filter:  525,957 rows
    PERMNOs: 227
    Dates: 2004-01-02 -> 2024-12-31
    Stocks/day: mean=99.5, min=97
    Factor columns: 170
    No duplicate (permno, date), no NaN in target/weight

  Loading Panel B (stock monthly, post-exclusion)...
    Before filter: 25,194 rows
    After filter:  25,194 rows
    PERMNOs: 227
    Dates: 2004-01-31 -> 2024-12-31
    Factor columns: 187
    No duplicate (permno, date)

  Exclusions verified: no ISO, no discontinued OAP, no ExchSwitch

PHASE 2: Z-SCORE PANEL B (MONTHLY) + FORWARD-FILL TO DAILY

  Z-scoring Panel B at monthly frequency...
    Robust expanding z-score: 187 features, 252 dates, 25,194 rows
    Warm-up: 1

In [12]:
import pandas as pd
import numpy as np

# ── Load both panels ──────────────────────────────────────────────────────────
print("Loading panels...")
panel_a = pd.read_parquet("../../Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering/panel_stock_daily_engineered.parquet")
panel_b = pd.read_parquet("../../Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering/panel_stock_monthly_engineered.parquet")
panel_a['date'] = pd.to_datetime(panel_a['date'])
panel_b['date'] = pd.to_datetime(panel_b['date'])

# Convert nullable dtypes
for df in [panel_a, panel_b]:
    for col in df.columns:
        if hasattr(df[col].dtype, 'numpy_dtype'):
            df[col] = df[col].astype('float64')

# ── Identify feature columns ─────────────────────────────────────────────────
a_non_feat = {'permno', 'date', 'dlyret', 'dlycap', 'year'}
b_non_feat = {'permno', 'date', 'month_end_cap', 'month_end_price', 'year', 'ExchSwitch'}

a_feat_cols = [c for c in panel_a.columns if c not in a_non_feat]
b_feat_cols = [c for c in panel_b.columns if c not in b_non_feat]

# ── Check each feature: would it produce |z| > 50? ───────────────────────────
# Proxy: max(|x - mean|) / std across all stock-date observations.
# If this ratio > 50, the feature will produce z-scores > 50.

print(f"\n{'=' * 90}")
print("PANEL A (DAILY) -- features with potential |z| > 50")
print(f"{'=' * 90}")
print(f"\n  {'Feature':<45} {'Max |z| proxy':>14} {'Std':>14} {'Min':>14} {'Max':>14} {'NaN%':>7}")
print(f"  {'-' * 100}")

a_suspects = []
for col in a_feat_cols:
    vals = panel_a[col].dropna().values
    if len(vals) < 100:
        continue
    mean = vals.mean()
    std = vals.std(ddof=1)
    if std < 1e-6:
        a_suspects.append((col, np.inf, std, vals.min(), vals.max(), panel_a[col].isna().mean() * 100))
        print(f"  {col:<45} {'inf (std≈0)':>14} {std:>14.8f} {vals.min():>14.4f} {vals.max():>14.4f} {panel_a[col].isna().mean()*100:>6.2f}%")
    else:
        max_z = np.max(np.abs(vals - mean)) / std
        if max_z > 50:
            nan_pct = panel_a[col].isna().mean() * 100
            a_suspects.append((col, max_z, std, vals.min(), vals.max(), nan_pct))
            print(f"  {col:<45} {max_z:>14.1f} {std:>14.4f} {vals.min():>14.4f} {vals.max():>14.4f} {nan_pct:>6.2f}%")

print(f"\n  Total suspects: {len(a_suspects)} / {len(a_feat_cols)}")

print(f"\n{'=' * 90}")
print("PANEL B (MONTHLY) -- features with potential |z| > 50")
print(f"{'=' * 90}")
print(f"\n  {'Feature':<45} {'Max |z| proxy':>14} {'Std':>14} {'Min':>14} {'Max':>14} {'NaN%':>7}")
print(f"  {'-' * 100}")

b_suspects = []
for col in b_feat_cols:
    vals = panel_b[col].dropna().values
    if len(vals) < 100:
        continue
    mean = vals.mean()
    std = vals.std(ddof=1)
    if std < 1e-6:
        b_suspects.append((col, np.inf, std, vals.min(), vals.max(), panel_b[col].isna().mean() * 100))
        print(f"  {col:<45} {'inf (std≈0)':>14} {std:>14.8f} {vals.min():>14.4f} {vals.max():>14.4f} {panel_b[col].isna().mean()*100:>6.2f}%")
    else:
        max_z = np.max(np.abs(vals - mean)) / std
        if max_z > 50:
            nan_pct = panel_b[col].isna().mean() * 100
            b_suspects.append((col, max_z, std, vals.min(), vals.max(), nan_pct))
            print(f"  {col:<45} {max_z:>14.1f} {std:>14.4f} {vals.min():>14.4f} {vals.max():>14.4f} {nan_pct:>6.2f}%")

print(f"\n  Total suspects: {len(b_suspects)} / {len(b_feat_cols)}")

# ── For each suspect, show top 5 worst observations ──────────────────────────
print(f"\n{'=' * 90}")
print("WORST OBSERVATIONS PER SUSPECT FEATURE")
print(f"{'=' * 90}")

for panel_name, suspects, df in [("Panel A", a_suspects, panel_a),
                                   ("Panel B", b_suspects, panel_b)]:
    for col, max_z, std, mn, mx, nan_pct in suspects[:20]:  # limit to first 20
        print(f"\n  {panel_name} / {col} (max |z| proxy: {max_z:.1f})")
        sub = df[['permno', 'date', col]].dropna()
        sub['abs_val'] = sub[col].abs()
        worst = sub.nlargest(5, 'abs_val')
        for _, row in worst.iterrows():
            print(f"    PERMNO {int(row['permno'])}  {str(row['date'].date())}  value={row[col]:.6f}")

Loading panels...

PANEL A (DAILY) -- features with potential |z| > 50

  Feature                                        Max |z| proxy            Std            Min            Max    NaN%
  ----------------------------------------------------------------------------------------------------
  dlyreti                                                124.2         0.0008         0.0000         0.1028   0.00%
  bid_ask_spread                                         478.2         0.0014         0.0000         0.6667   0.00%
  dollarpriceimpact_lr_ave                               198.3         0.1165       -19.4653        23.1237   2.23%
  dollarpriceimpact_lr_dw                                207.9         0.1315       -19.3711        27.3849   2.23%
  dollarpriceimpact_lr_sw                                207.5         0.1315       -19.3711        27.3154   2.23%
  dollarrealizedspread_lr_ave                            297.8         0.1284       -19.9709        38.2436   2.23%
  dollarreali

In [14]:
import pandas as pd
import numpy as np

# Load the z-scored Panel A from memory, or re-run just the z-score step
# If panel_a_z is not in memory, this standalone version works:
panel_a = pd.read_parquet("../../Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering/panel_stock_daily_engineered.parquet")
panel_a['date'] = pd.to_datetime(panel_a['date'])

a_non_feat = {'permno', 'date', 'dlyret', 'dlycap', 'year'}
feat_cols = [c for c in panel_a.columns if c not in a_non_feat]

# Quick cross-sectional z-score for comparison
# For each date: z_it = (x_it - mean_t) / std_t
print("Computing cross-sectional per-date z-scores for comparison...")
grouped = panel_a.groupby('date')
cs_mean = grouped[feat_cols].transform('mean')
cs_std = grouped[feat_cols].transform('std')
cs_std = cs_std.replace(0, np.nan)
cs_z = (panel_a[feat_cols] - cs_mean) / cs_std

print(f"\nCross-sectional per-date z-score stats:")
per_feat_std = cs_z.std()
print(f"  Per-feature std: mean={per_feat_std.mean():.4f}, "
      f"min={per_feat_std.min():.4f}, max={per_feat_std.max():.4f}")
print(f"  Features with std > 2: {(per_feat_std > 2).sum()}")
print(f"  Features with std < 0.5: {(per_feat_std < 0.5).sum()}")
print(f"  Features with std in [0.8, 1.2]: {((per_feat_std >= 0.8) & (per_feat_std <= 1.2)).sum()}")
# Convert to numpy-safe float first
cs_vals = cs_z[feat_cols].astype('float64').values
print(f"  Overall std: {np.nanstd(cs_vals):.4f}")
print(f"  Overall max |z|: {np.nanmax(np.abs(cs_vals)):.2f}")

Computing cross-sectional per-date z-scores for comparison...

Cross-sectional per-date z-score stats:
  Per-feature std: mean=0.9948, min=0.9941, max=0.9950
  Features with std > 2: 0
  Features with std < 0.5: 0
  Features with std in [0.8, 1.2]: 192
  Overall std: 0.9948
  Overall max |z|: 9.90
